# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides a complete walkthrough for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the `@id` property to reference all key entities in the dataset--record sets, fields, and columns.

In [ ]:
# Retrieve and display all record sets and their fields by @id
print("Record Sets available in this dataset:")
record_set_ids = []
for record_set in metadata.record_set:
    print(f"  - {record_set['@id']} : {record_set.get('name', '')}")
    record_set_ids.append(record_set['@id'])
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("    Fields:")
    for field in fields:
        if isinstance(field, dict):
            f_id = field.get('@id', '')
            fname = field.get('name', '')
        else:  # it may be an @id string
            f_id = field
            fname = ''
        print(f"      - {f_id} {fname}")
if not record_set_ids:
    print("No record sets found. Attempting to infer them by loading records...")
    # Try to find any available record_set from the data API
    # mlcroissant falls back to the first if not specified
    records_preview = dataset.records()
    sample = next(records_preview)
    print("Fields in the default record set:")
    for key in sample:
        print(f"  - {key}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use the record set and field `@id`s from above.

Below we extract the data for each detected record set (or the default if only one is available).

In [ ]:
# Extract data from each available record set
dataframes = {}
used_record_set_ids = record_set_ids

if not used_record_set_ids:
    # If no record_set in metadata, use default None
    print("No explicit record sets found. Using the default (None) record set.")
    records_gen = dataset.records()
    records = list(records_gen)
    df = pd.DataFrame(records)
    dataframes[None] = df
    print(f"Available columns: {df.columns.tolist()}")
    display_df_key = None
else:
    for record_set_id in used_record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    display_df_key = used_record_set_ids[0]
    print(f"Available columns in '{display_df_key}': {dataframes[display_df_key].columns.tolist()}")
dataframes[display_df_key].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes to prepare the dataset for further analysis.

All fields are referenced by their `@id` as specified in the metadata.

In [ ]:
# Example EDA: Filter, Normalize, and Group by @id
df = dataframes[display_df_key]
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields detected: {numeric_candidates}")

# Choose a numeric field for analysis
if numeric_candidates:
    numeric_field = numeric_candidates[0]  # or pick a specific known field, e.g. 'Coverage (%)' by its header
else:
    print("No numeric columns found. Aborting EDA example.")
    numeric_field = None

if numeric_field:
    threshold = df[numeric_field].mean()  # use mean as example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df[[numeric_field]].head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a likely categorical field (e.g. 'Description' or 'Sample Name')
    object_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field = None
    for field in object_candidates:
        # Try to pick a usable group field (by unique count)
        if df[field].nunique() < 20:
            group_field = field
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No suitable group field (categorical, <20 unique values) found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is a histogram for a selected numeric field, and a boxplot comparing grouped data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # Boxplot (if group_field was selected above)
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} distribution grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook provided a step-by-step exploration of the FAIR² dataset using the `mlcroissant` library. We loaded metadata, reviewed available fields and record sets via their `@id`, performed basic data extraction, filtering, normalization, grouping, and visualized selected properties. For deeper analysis, consult the dataset documentation and extend this notebook according to your research questions.